# ML Tutorial: Fixing Overfitting & Proper Model Evaluation

## Learning Objectives
1. Understand what overfitting is and why it matters for business
2. Use cross-validation instead of simple train/test split
3. Evaluate models properly with confusion matrix and classification reports
4. Understand feature importance
5. Visualize learning curves to detect overfitting

**Business Context:** These techniques are crucial for deploying ML in production (like at Vultun!)

## Step 1: Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, learning_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully!")

In [ ]:
# Load data
df = pd.read_csv('../Data/daily_summary.csv')
df = df.dropna(axis=0, how='all')
df = df.drop('index', axis=1, errors='ignore')

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

## Step 2: Check Class Balance

**Why this matters:** Imbalanced classes (e.g., 70% 'up' days, 30% 'down' days) can make your model biased.

In [ ]:
# Check class distribution
class_counts = df['s&p_up/down'].value_counts()
print("Class Distribution:")
print(class_counts)
print(f"\nPercentages:")
print(class_counts / len(df) * 100)

# Visualize
plt.figure(figsize=(8, 5))
class_counts.plot(kind='bar', color=['green', 'red'])
plt.title('Distribution of S&P 500 Up/Down Days', fontsize=14)
plt.xlabel('Direction')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

# Is it balanced?
up_pct = (class_counts['up'] / len(df)) * 100
if 45 <= up_pct <= 55:
    print("\n✅ Classes are balanced!")
else:
    print(f"\n⚠️ Classes are imbalanced! {up_pct:.1f}% up days. We should handle this.")

## Step 3: Prepare Features and Labels

In [ ]:
# Select features (X) and label (y)
features_to_drop = ['date', 's&p_up/down', 's&p_%change', 'isRetweet', 'isDeleted', 'negative', 'neutral']
X = df.drop(features_to_drop, axis=1)
y = df['s&p_up/down']

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"\nFeatures we're using:")
print(list(X.columns))

## Step 4: Split Data (70% train, 30% test)

**Important:** We use `stratify=y` to maintain class balance in both train and test sets.

In [ ]:
# Split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3, 
    random_state=42,  # For reproducibility
    stratify=y        # Maintain class balance
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining set class distribution:")
print(y_train.value_counts())
print(f"\nTest set class distribution:")
print(y_test.value_counts())

## Step 5: Scale the Features

**Why?** Features like 'retweets' (500,000) and 'tweets' (5) have very different scales. Scaling puts them on equal footing.

In [ ]:
# Fit scaler on training data only!
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Use same scaler, don't fit again!

print("✅ Features scaled to [0, 1] range")
print(f"\nExample - first sample before scaling:")
print(X_train.iloc[0].values)
print(f"\nExample - first sample after scaling:")
print(X_train_scaled[0])

## Step 6: Train Models - Compare Simple vs Complex

Let's train TWO models to see overfitting in action:
- **Model A:** Simple (max_depth=3) - Should generalize well
- **Model B:** Complex (max_depth=10) - Might overfit

In [ ]:
# Model A: Simple (Less Overfitting)
model_simple = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    min_samples_split=10,
    random_state=42,
    class_weight='balanced'  # Handle class imbalance
)

# Model B: Complex (More Overfitting Risk)
model_complex = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=2,
    random_state=42,
    class_weight='balanced'
)

# Train both models
print("Training Model A (Simple)...")
model_simple.fit(X_train_scaled, y_train)
print("✅ Model A trained")

print("\nTraining Model B (Complex)...")
model_complex.fit(X_train_scaled, y_train)
print("✅ Model B trained")

## Step 7: Compare Train vs Test Performance

**Large gap = Overfitting!**

In [ ]:
# Model A (Simple)
train_score_simple = model_simple.score(X_train_scaled, y_train)
test_score_simple = model_simple.score(X_test_scaled, y_test)

# Model B (Complex)
train_score_complex = model_complex.score(X_train_scaled, y_train)
test_score_complex = model_complex.score(X_test_scaled, y_test)

# Display results
print("="*60)
print("MODEL A (Simple: max_depth=3)")
print("="*60)
print(f"Training Accuracy:   {train_score_simple:.4f} ({train_score_simple*100:.2f}%)")
print(f"Test Accuracy:       {test_score_simple:.4f} ({test_score_simple*100:.2f}%)")
print(f"Gap (Overfitting):   {(train_score_simple - test_score_simple)*100:.2f}%")

print("\n" + "="*60)
print("MODEL B (Complex: max_depth=10)")
print("="*60)
print(f"Training Accuracy:   {train_score_complex:.4f} ({train_score_complex*100:.2f}%)")
print(f"Test Accuracy:       {test_score_complex:.4f} ({test_score_complex*100:.2f}%)")
print(f"Gap (Overfitting):   {(train_score_complex - test_score_complex)*100:.2f}%")

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
models = ['Simple\n(max_depth=3)', 'Complex\n(max_depth=10)']
train_scores = [train_score_simple, train_score_complex]
test_scores = [test_score_simple, test_score_complex]

x = np.arange(len(models))
width = 0.35

bars1 = ax.bar(x - width/2, train_scores, width, label='Training', color='skyblue')
bars2 = ax.bar(x + width/2, test_scores, width, label='Test', color='orange')

ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Training vs Test Accuracy: Simple vs Complex Model', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim([0, 1])

# Add value labels on bars
for bar in bars1 + bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("\n💡 Analysis:")
if (train_score_complex - test_score_complex) > 0.15:
    print("   Complex model is OVERFITTING! Big gap between train and test.")
if (train_score_simple - test_score_simple) < 0.1:
    print("   Simple model generalizes better! Smaller gap = more reliable.")

## Step 8: Cross-Validation - The Professional Way

**Problem with train/test split:** You might get lucky/unlucky with the split.

**Solution:** Cross-validation tests on multiple splits!

```
5-Fold Cross-Validation:
Fold 1: [Train Train Train Train | Test]
Fold 2: [Train Train Train | Test | Train]
Fold 3: [Train Train | Test | Train Train]
Fold 4: [Train | Test | Train Train Train]
Fold 5: [Test | Train Train Train Train]
```

Average all 5 test scores = More reliable estimate!

In [ ]:
# Set up 5-fold cross-validation with stratification
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Cross-validate Model A (Simple)
print("Running 5-fold cross-validation on Model A (Simple)...")
cv_scores_simple = cross_val_score(
    model_simple, 
    X_train_scaled, 
    y_train, 
    cv=cv, 
    scoring='accuracy'
)

print(f"\nFold scores: {cv_scores_simple}")
print(f"Mean Accuracy: {cv_scores_simple.mean():.4f} ({cv_scores_simple.mean()*100:.2f}%)")
print(f"Std Deviation: {cv_scores_simple.std():.4f} (±{cv_scores_simple.std()*100:.2f}%)")

# Cross-validate Model B (Complex)
print("\n" + "="*60)
print("Running 5-fold cross-validation on Model B (Complex)...")
cv_scores_complex = cross_val_score(
    model_complex, 
    X_train_scaled, 
    y_train, 
    cv=cv, 
    scoring='accuracy'
)

print(f"\nFold scores: {cv_scores_complex}")
print(f"Mean Accuracy: {cv_scores_complex.mean():.4f} ({cv_scores_complex.mean()*100:.2f}%)")
print(f"Std Deviation: {cv_scores_complex.std():.4f} (±{cv_scores_complex.std()*100:.2f}%)")

# Visualize CV scores
fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot([cv_scores_simple, cv_scores_complex], 
                 labels=['Simple Model', 'Complex Model'],
                 patch_artist=True)

# Color the boxes
colors = ['lightblue', 'lightcoral']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_ylabel('Cross-Validation Accuracy', fontsize=12)
ax.set_title('5-Fold Cross-Validation Scores Distribution', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.show()

print("\n💡 Business Insight:")
print(f"   If you deployed Model A, expect ~{cv_scores_simple.mean()*100:.1f}% accuracy in production.")
print(f"   Lower std deviation = more consistent performance!")

## Step 9: Detailed Evaluation - Confusion Matrix

**Accuracy alone isn't enough!** What if your model just predicts "up" every time?

Confusion Matrix shows:
- **True Positives:** Correctly predicted "up"
- **True Negatives:** Correctly predicted "down"
- **False Positives:** Predicted "up" but was "down" (Type I error)
- **False Negatives:** Predicted "down" but was "up" (Type II error)

In [ ]:
# Make predictions on test set
y_pred_simple = model_simple.predict(X_test_scaled)
y_pred_complex = model_complex.predict(X_test_scaled)

# Confusion matrices
cm_simple = confusion_matrix(y_test, y_pred_simple)
cm_complex = confusion_matrix(y_test, y_pred_complex)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Simple model
sns.heatmap(cm_simple, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['down', 'up'], yticklabels=['down', 'up'])
axes[0].set_title('Simple Model - Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Actual', fontsize=12)
axes[0].set_xlabel('Predicted', fontsize=12)

# Complex model
sns.heatmap(cm_complex, annot=True, fmt='d', cmap='Reds', ax=axes[1],
            xticklabels=['down', 'up'], yticklabels=['down', 'up'])
axes[1].set_title('Complex Model - Confusion Matrix', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Actual', fontsize=12)
axes[1].set_xlabel('Predicted', fontsize=12)

plt.tight_layout()
plt.show()

print("How to read the confusion matrix:")
print("- Top-left (True Negative): Correctly predicted 'down'")
print("- Bottom-right (True Positive): Correctly predicted 'up'")
print("- Top-right (False Positive): Wrongly predicted 'up'")
print("- Bottom-left (False Negative): Wrongly predicted 'down'")

## Step 10: Classification Report

Shows:
- **Precision:** When model says "up", how often is it right?
- **Recall:** Of all actual "up" days, how many did we catch?
- **F1-Score:** Harmonic mean of precision and recall

In [ ]:
print("="*60)
print("SIMPLE MODEL - Classification Report")
print("="*60)
print(classification_report(y_test, y_pred_simple))

print("\n" + "="*60)
print("COMPLEX MODEL - Classification Report")
print("="*60)
print(classification_report(y_test, y_pred_complex))

print("\n💡 Business Translation:")
print("   Precision: If model predicts 'up', what's the probability it's correct?")
print("   Recall: Of all 'up' days, what % did we predict correctly?")
print("   F1-Score: Balance between precision and recall (higher is better)")

## Step 11: Feature Importance

**Which features actually matter?**

This is GOLD for business - tells you what to focus on!

In [ ]:
# Get feature importances from the simple model
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model_simple.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importance Rankings:")
print(feature_importance)

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='steelblue')
plt.xlabel('Importance', fontsize=12)
plt.title('Feature Importance - Simple Model', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\n💡 Business Insight:")
print(f"   Top 3 most important features:")
for i, row in feature_importance.head(3).iterrows():
    print(f"   {i+1}. {row['feature']}: {row['importance']:.4f}")
print("\n   Focus on these features for better predictions!")

## Step 12: Learning Curves - Visualize Overfitting

**This is the BEST way to diagnose overfitting!**

Shows how performance changes as you add more training data.

In [ ]:
def plot_learning_curve(model, X, y, title):
    """Plot learning curve to visualize overfitting"""
    train_sizes, train_scores, test_scores = learning_curve(
        model, X, y, 
        cv=5, 
        n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 10),
        random_state=42
    )
    
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)
    test_std = np.std(test_scores, axis=1)
    
    plt.figure(figsize=(10, 6))
    plt.plot(train_sizes, train_mean, label='Training score', color='blue', marker='o')
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    
    plt.plot(train_sizes, test_mean, label='Cross-validation score', color='orange', marker='s')
    plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color='orange')
    
    plt.xlabel('Training Set Size', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.legend(loc='best', fontsize=11)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

# Plot for simple model
print("Generating learning curve for Simple Model...")
plot_learning_curve(model_simple, X_train_scaled, y_train, 
                    'Learning Curve - Simple Model (max_depth=3)')

print("\nGenerating learning curve for Complex Model...")
plot_learning_curve(model_complex, X_train_scaled, y_train,
                    'Learning Curve - Complex Model (max_depth=10)')

print("\n💡 How to Read Learning Curves:")
print("   - Large gap between blue and orange = OVERFITTING")
print("   - Lines converging = Good generalization")
print("   - Both lines flat and low = Underfitting (model too simple)")
print("   - Both lines increasing with more data = More data might help!")

## Summary & Business Recommendations

### What We Learned

1. **Overfitting Detection:**
   - Large train/test gap = Overfitting
   - Use cross-validation for honest estimates
   - Learning curves visualize the problem

2. **Model Selection:**
   - Simpler models (max_depth=3) generalize better
   - Complex models (max_depth=10) memorize training data
   - Use `class_weight='balanced'` for imbalanced data

3. **Evaluation Metrics:**
   - Accuracy isn't everything
   - Check precision, recall, F1-score
   - Confusion matrix shows types of errors

4. **Feature Importance:**
   - Focus on features that actually matter
   - Drop low-importance features to reduce noise

### For Vultun (Your Business)

**Before deploying ANY ML model:**

✅ Use cross-validation (not just train/test split)

✅ Check confusion matrix (understand your errors)

✅ Plot learning curves (diagnose overfitting)

✅ Test on REAL unseen data before production

✅ Monitor performance in production (it might drift!)

✅ Set realistic expectations (60% accuracy is honest if that's what CV shows)

### Next Steps

1. Try different models (XGBoost, LightGBM)
2. Feature engineering (create new features)
3. More data (your model might improve with more training samples)
4. Ensemble methods (combine multiple models)
5. Time-based validation (for stock market, respect time order!)

## Bonus: Save Your Best Model

Once you're happy with the model, save it for deployment!

In [ ]:
import joblib

# Save the model and scaler
model_filename = 'best_model_random_forest.pkl'
scaler_filename = 'feature_scaler.pkl'

joblib.dump(model_simple, model_filename)
joblib.dump(scaler, scaler_filename)

print(f"✅ Model saved to: {model_filename}")
print(f"✅ Scaler saved to: {scaler_filename}")
print("\nTo load later:")
print("  model = joblib.load('best_model_random_forest.pkl')")
print("  scaler = joblib.load('feature_scaler.pkl')")